In [1]:
import json
import sys
import pandas as pd

sys.path.append("../src")
import metrics
import sec_extract


def search_period_tags(cik, fiscal_year, keywords):
    """List full-year 10-K tags (income or cash flow) whose names contain any keyword, in $000s."""
    path = f"../data/raw/companyfacts/CIK{int(cik):010d}.json"
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    found = set()
    for tag, info in data["facts"].get("us-gaap", {}).items():
        if any(k in tag for k in keywords) and "USD" in info["units"]:
            for e in info["units"]["USD"]:
                if e.get("form") == "10-K" and "start" in e:          # has a start date = a period
                    start, end = pd.Timestamp(e["start"]), pd.Timestamp(e["end"])
                    full_year = 350 <= (end - start).days <= 380
                    same_year = (end - pd.Timedelta(days=15)).year == fiscal_year
                    if full_year and same_year:
                        found.add((tag, e["val"] / 1000))
    return sorted(found)


all_metrics = pd.read_csv("../data/processed/metrics.csv")
qnst_cik = all_metrics.loc[all_metrics["ticker"] == "QNST", "cik"].iloc[0]

for tag, value in search_period_tags(qnst_cik, 2026, ["PaymentsToAcquire", "BusinessCombinationConsideration"]):
    print(f"{tag:80} {value:>12,.0f}")

PaymentsToAcquirePropertyPlantAndEquipment                                              3,397


In [2]:
for tag, value in search_period_tags(qnst_cik, 2026, ["Acqui", "Business"]):
    print(f"{tag:80} {value:>12,.0f}")

BusinessCombinationContingentConsiderationArrangementsChangeInAmountOfContingentConsiderationLiability1        4,650
GoodwillAcquiredDuringPeriod                                                          136,742
NoncashOrPartNoncashAcquisitionValueOfLiabilitiesAssumed1                                   0
PaymentsToAcquirePropertyPlantAndEquipment                                              3,397
TreasuryStockValueAcquiredCostMethod                                                   31,442


In [3]:
import scoring

all_metrics = pd.read_csv("../data/processed/metrics.csv")
valuation = pd.read_csv("../data/processed/valuation.csv")

# Companies kept after the manual review (docs/manual_review.md)
REVIEWED = ["WLDN", "QNST", "LQDT", "ONT", "TISI", "ADV"]

with_cash_conv = scoring.add_three_year_cash_conversion(all_metrics)
latest = valuation[valuation["ticker"].isin(REVIEWED)][["cik", "ticker", "fiscal_year", "ev_to_ebitda"]]
candidates = latest.merge(with_cash_conv.drop(columns="ticker"), on=["cik", "fiscal_year"])

ranked = scoring.score_companies(candidates)

score_cols = [f"score_{m}" for m in scoring.SCORING]
ranked[["ticker", "eligible", "total_score"] + score_cols].round(1)

,ticker,eligible,total_score,score_revenue_cagr_3y,score_ebitda_margin,score_fcf_conversion_3y,score_capex_intensity,score_net_leverage,score_ev_to_ebitda
2,QNST,True,66.4,100.0,0.0,100.0,95.6,100.0,34.0
1,WLDN,True,64.7,100.0,28.1,100.0,77.0,100.0,6.7
0,LQDT,True,63.3,100.0,30.3,100.0,72.7,100.0,0.0
3,ONT,False,59.6,100.0,16.9,100.0,67.3,0.0,47.6
5,TISI,False,32.9,21.8,2.5,0.0,82.7,0.0,99.0
4,ADV,False,29.7,0.0,0.0,100.0,97.0,0.0,0.0


In [4]:
weight_scenarios = {
    "Base case":         {"revenue_cagr_3y": 20, "ebitda_margin": 20, "fcf_conversion_3y": 20,
                          "capex_intensity": 10, "net_leverage": 10, "ev_to_ebitda": 20},
    "Equal weights":     {m: 1 for m in scoring.SCORING},
    "Quality focus":     {"revenue_cagr_3y": 10, "ebitda_margin": 35, "fcf_conversion_3y": 30,
                          "capex_intensity": 10, "net_leverage": 5, "ev_to_ebitda": 10},
    "Value focus":       {"revenue_cagr_3y": 15, "ebitda_margin": 15, "fcf_conversion_3y": 15,
                          "capex_intensity": 5, "net_leverage": 10, "ev_to_ebitda": 40},
    "Growth discounted": {"revenue_cagr_3y": 5, "ebitda_margin": 25, "fcf_conversion_3y": 25,
                          "capex_intensity": 10, "net_leverage": 10, "ev_to_ebitda": 25},
}

rows = []
for name, weights in weight_scenarios.items():
    # Same scales as the base model; only the weights change
    config = {m: (zero, full, weights[m]) for m, (zero, full, _) in scoring.SCORING.items()}
    result = scoring.score_companies(candidates, scoring=config)
    eligible = result[result["eligible"]]
    rows.append({"scenario": name,
                 "ranking (eligible only)": " > ".join(eligible["ticker"]),
                 "scores": ", ".join(f"{s:.1f}" for s in eligible["total_score"]),
                 "gap #1 vs #2": round(eligible["total_score"].iloc[0] - eligible["total_score"].iloc[1], 1)})

sensitivity = pd.DataFrame(rows)
sensitivity.to_csv("../data/processed/scoring_sensitivity.csv", index=False)
sensitivity

,scenario,ranking (eligible only),scores,gap #1 vs #2
0,Base case,QNST > WLDN > LQDT,"66.4, 64.7, 63.3",1.7
1,Equal weights,QNST > WLDN > LQDT,"71.6, 68.6, 67.2",3.0
2,Quality focus,WLDN > LQDT > QNST,"63.2, 62.9, 58.0",0.4
3,Value focus,QNST > WLDN > LQDT,"58.4, 50.7, 48.2",7.6
4,Growth discounted,QNST > WLDN > LQDT,"58.1, 56.4, 54.8",1.7
